## Setup Koneksi & Environmetn

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from supabase import create_client, Client
from pathlib import Path
import os
from dotenv import load_dotenv

# Dapatkan lokasi folder saat notebook ini berjalan
_notebook_dir = Path.cwd()

# Menyesuaikan path .env berdasarkan posisi notebook
if _notebook_dir.name == "notebooks":
    env_path = _notebook_dir.parent / ".env"
else:
    env_path = _notebook_dir / "backend" / ".env"

load_dotenv(env_path)

# Inisialisasi Supabase Client
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    print("Error: Kredensial Supabase tidak ditemukan di file .env")
else:
    supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
    print("Koneksi Supabase berhasil diinisiasi!")

Koneksi Supabase berhasil diinisiasi!


## Mengambil Data Time-Series dari Supabase

In [2]:
TARGET_CARD_ID = "base1-4" 

try:
    # 1. Mengambil data menggunakan API Supabase dan mengurutkan berdasarkan recorded_date
    response = supabase.table("card_price_history") \
        .select("*") \
        .eq("card_id", TARGET_CARD_ID) \
        .order("recorded_date") \
        .execute()
    
    # 2. Konversi hasil response langsung ke Pandas DataFrame
    df = pd.DataFrame(response.data)
    
    # 3. Preprocessing khusus untuk time-series menggunakan 'recorded_date'
    df['recorded_date'] = pd.to_datetime(df['recorded_date'])
    df = df.set_index('recorded_date')
    
    print(f"Berhasil memuat {len(df)} baris data historis untuk kartu {TARGET_CARD_ID}.")
    display(df.head())
    
except Exception as e:
    print(f"Gagal mengambil data: {e}")

Berhasil memuat 3 baris data historis untuk kartu base1-4.


,card_id,tcg_normal_market,tcg_normal_low,tcg_normal_mid,tcg_normal_high,tcg_holo_market,tcg_holo_low,tcg_holo_mid,tcg_holo_high,tcg_reverse_market,cardmarket_trend,cardmarket_avg_sell,cardmarket_low,cardmarket_avg1,cardmarket_avg7,cardmarket_avg30,effective_market_price,created_at
recorded_date,,,,,,,,,,,,,,,,,,
2026-09-22,base1-4,None,None,None,None,919.98,449.99,849.55,3499.1,None,4184.6,1531.0,799.0,14950.0,3939.88,2427.79,919.98,2026-09-22T17:02:43.367977+00:00
2026-09-25,base1-4,None,None,None,None,944.53,449.99,839.99,3499.1,None,4184.6,1531.0,799.0,14950.0,3939.88,2427.79,944.53,2026-09-25T04:23:14.584012+00:00
2026-09-28,base1-4,None,None,None,None,944.53,499.99,832.49,3499.1,None,4184.6,1531.0,799.0,14950.0,3939.88,2427.79,944.53,2026-09-28T04:46:51.709505+00:00


### Cek Data

In [4]:
# ── Load SEMUA data (Supabase membatasi 1000 baris per request → pakai pagination)
def fetch_all(table, columns="*", page=1000):
    rows, start = [], 0
    while True:
        res = (supabase.table(table).select(columns)
               .order("recorded_date").order("card_id")
               .range(start, start + page - 1).execute())
        rows.extend(res.data)
        if len(res.data) < page:
            break
        start += page
    return pd.DataFrame(rows)

df_all = fetch_all(
    "card_price_history",
    "card_id, recorded_date, cardmarket_avg_sell, cardmarket_trend, "
    "cardmarket_avg7, cardmarket_avg30, effective_market_price"
)
df_all["recorded_date"] = pd.to_datetime(df_all["recorded_date"])

# ── Diagnostik kelayakan trend/forecast
per_card = df_all.groupby("card_id")["recorded_date"].nunique()

print(f"Total baris     : {len(df_all):,}")
print(f"Kartu unik      : {df_all['card_id'].nunique():,}")
print(f"Hari unik       : {df_all['recorded_date'].nunique()}  "
      f"({df_all['recorded_date'].min().date()} → {df_all['recorded_date'].max().date()})")
print(f"Duplikat (kartu+tanggal): {df_all.duplicated(['card_id','recorded_date']).sum():,}")
print()
print("Hari data per kartu:")
print(per_card.describe().round(1))
print()
print(f"Kartu ≥ 3 hari  : {(per_card >= 3).sum():,}   (linear trend)")
print(f"Kartu ≥ 7 hari  : {(per_card >= 7).sum():,}   (Holt-Winters / SMA-7 bermakna)")
print(f"Kartu ≥ 14 hari : {(per_card >= 14).sum():,}   (Prophet)")
print(f"Kartu ≥ 30 hari : {(per_card >= 30).sum():,}   (SMA-30 bermakna, backtest MAPE)")

Total baris     : 88,964
Kartu unik      : 20,617
Hari unik       : 5  (2026-09-22 → 2026-09-28)
Duplikat (kartu+tanggal): 0

Hari data per kartu:
count    20617.0
mean         4.3
std          0.8
min          2.0
25%          4.0
50%          4.0
75%          5.0
max          5.0
Name: recorded_date, dtype: float64

Kartu ≥ 3 hari  : 20,367   (linear trend)
Kartu ≥ 7 hari  : 0   (Holt-Winters / SMA-7 bermakna)
Kartu ≥ 14 hari : 0   (Prophet)
Kartu ≥ 30 hari : 0   (SMA-30 bermakna, backtest MAPE)


## Logika Forecasting & Sinyal Rekomendasi Buy, Hold, Sell

In [3]:
# 0. Data Cleaning: Mengisi nilai NULL (NaN) dengan harga di hari sebelumnya
df['effective_market_price'] = df['effective_market_price'].ffill()

# Jika di awal data masih ada NULL, isi dengan 0 atau buang (opsional)
df = df.dropna(subset=['effective_market_price'])

# 1. Kalkulasi Moving Average menggunakan 'effective_market_price'
df['SMA_7'] = df['effective_market_price'].rolling(window=7, min_periods=1).mean()
df['SMA_30'] = df['effective_market_price'].rolling(window=30, min_periods=1).mean()

# 2. Logika Sinyal Arbitrase (BUY / HOLD / SELL)
def generate_signal(row):
    if row['SMA_7'] > row['SMA_30']:
        return "BUY"
    elif row['SMA_7'] < row['SMA_30']:
        return "SELL"
    else:
        return "HOLD"

df['signal'] = df.apply(generate_signal, axis=1)

# 3. Visualisasi Interaktif menggunakan Plotly
fig = go.Figure()

# Plot Harga Asli
fig.add_trace(go.Scatter(x=df.index, y=df['effective_market_price'], mode='lines', name='Harga Harian (Market)', line=dict(color='blue')))
# Plot SMA 7
fig.add_trace(go.Scatter(x=df.index, y=df['SMA_7'], mode='lines', name='Tren Jangka Pendek (7 Hari)', line=dict(color='orange', dash='dot')))
# Plot SMA 30
fig.add_trace(go.Scatter(x=df.index, y=df['SMA_30'], mode='lines', name='Tren Jangka Panjang (30 Hari)', line=dict(color='red', dash='dot')))

fig.update_layout(
    title=f"Forecasting & Analitik Sinyal Trading untuk {TARGET_CARD_ID}",
    xaxis_title="Tanggal",
    yaxis_title="Harga ($)",
    hovermode="x unified"
)

fig.show()

# Menampilkan status sinyal paling akhir (terbaru)
if not df.empty:
    print("Status Sinyal Hari Ini:", df.iloc[-1]['signal'])
else:
    print("Tidak ada data valid untuk menghasilkan sinyal.")

Status Sinyal Hari Ini: HOLD
